# GAN 评估实战 · Generative Dog Images

本 notebook 演示如何用 `gan_eval` 评估你的 GAN 生成结果。

**评估流程**：生成图 → 提特征 → 算 FID / MiFID / KID / IS / Precision-Recall → 看曲线与最近邻。

> 关键：GAN 的 loss 没有意义，**FID 曲线才是判断“改动是否有效”的唯一依据**。

In [ ]:
import os, sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# 让 notebook 能 import 到 gan_eval
PROJECT = r"C:\Users\moneyforever\Desktop\Deep-Learning\Kaggle"
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from gan_eval import GANEvaluator, FIDTracker, list_images, format_results

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch :", torch.__version__)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

## 1. 路径配置

- `REAL_DIR`：真实狗图目录（`data/all-dogs`）
- `FAKE_DIR`：某次生成结果的目录（训练脚本保存的图片）
- `OUT_DIR`：评估输出（json / 曲线 / 最近邻图）

In [ ]:
REAL_DIR   = os.path.join(PROJECT, "data", "all-dogs")
FAKE_DIR   = os.path.join(PROJECT, "outputs", "generated")
OUT_DIR    = os.path.join(PROJECT, "outputs", "eval")
IMAGE_SIZE = 64          # 比赛生成图是 64x64
os.makedirs(OUT_DIR, exist_ok=True)

for name, p in [("REAL_DIR", REAL_DIR), ("FAKE_DIR", FAKE_DIR)]:
    ok = os.path.isdir(p)
    n = len(list_images(p)) if ok else 0
    print(f"{name}: exists={ok}  images={n}  -> {p}")

## 2. 创建评估器

`GANEvaluator` 会**一次性**算好真实图的特征并缓存（真实集不变，不必重复计算）。
首次运行会加载 InceptionV3 权重（来自 download.pytorch.org，已缓存在本地）。

In [ ]:
if os.path.isdir(REAL_DIR) and len(list_images(REAL_DIR)) > 0:
    evaluator = GANEvaluator(real_dir=REAL_DIR, image_size=IMAGE_SIZE,
                             batch_size=64, num_workers=0)
    print("evaluator ready on", evaluator.device)
    _ = evaluator.real_features            # 触发真实特征计算 + 缓存
    print("real features cached:", evaluator.real_features.shape)
else:
    evaluator = None
    print("还没有真实数据。请先准备真实数据到 data/all-dogs。")

## 3. 评估一批生成图

需要先把生成图存成文件夹（例如训练时每 N 个 epoch 保存一次）。

In [ ]:
if evaluator is not None and os.path.isdir(FAKE_DIR) and len(list_images(FAKE_DIR)) > 0:
    scores = evaluator.evaluate(FAKE_DIR, metrics=("fid", "mifid", "kid", "is", "pr"))
    print(format_results(scores))
    with open(os.path.join(OUT_DIR, "last_eval.json"), "w", encoding="utf-8") as f:
        json.dump(scores, f, indent=2)
else:
    print("跳过：没有生成图文件夹（或还没数据）。")

## 4. 可视化：样本网格 + 最近邻查抄袭

- **样本网格**：肉眼快速判断质量。
- **最近邻**：上排 = 生成图，下排 = 特征空间里最像的真实图。
  如果两排几乎一模一样 → 模型在**记忆训练集**，MiFID 会惩罚。

In [ ]:
if evaluator is not None and os.path.isdir(FAKE_DIR) and len(list_images(FAKE_DIR)) > 0:
    nn_path = os.path.join(OUT_DIR, "nearest_neighbours.png")
    evaluator.nearest_neighbours(FAKE_DIR, nn_path, n=8)
    display(Image.open(nn_path))
else:
    print("跳过：没有生成图文件夹。")

## 5. 训练时跟踪 FID 曲线（核心）

GAN 的判别器/生成器 loss 几乎无法反映质量，**必须定期算 FID**。
`FIDTracker` 帮你：定期采样 → 存网格图 → 算分 → 写 json → 画曲线。

下面用一个**玩具生成器**演示（真实训练时替换成你的 G）。

In [ ]:
if evaluator is not None:
    tracker = FIDTracker(evaluator, out_dir=OUT_DIR, every=1, num_samples=256,
                         seed=0, metrics=("fid", "mifid", "kid", "is"))

    # ---- 玩具生成器：真实训练时替换成你的 G(z) ----
    def sample_fn(n, seed):
        g = torch.Generator().manual_seed(seed)
        z = torch.randn(n, 128, generator=g)
        w = torch.randn(128, 3 * 64 * 64, generator=g)
        imgs = torch.sigmoid(z @ w).reshape(n, 3, 64, 64)   # 仅演示流程
        return (imgs * 255).to(torch.uint8)

    for epoch in range(1, 6):
        tracker.step(epoch, sample_fn)

    display(Image.open(tracker.plot()))
    print("最佳 epoch（按 FID）:", tracker.best_epoch("fid"))
else:
    print("跳过：先准备真实数据。")

## 6. 怎么判读

| 现象 | 结论 |
|---|---|
| FID 随 epoch 持续下降 | 模型在进步 |
| FID 还行，但 Recall 很低 | **模式坍塌**（生成图都差不多） |
| MiFID 远大于 FID | 触发了记忆惩罚，模型在抄训练图 |
| FID 抖动很大 | 样本太少 / 训练不稳定，加大 num_samples |

**铁律**：只用同一个工具、同一份真实集做**相对比较**；不要拿本库的 FID 去和别人的数值比。

In [ ]:
if evaluator is not None and len(tracker.history) > 0:
    import pandas as pd
    df = pd.DataFrame(tracker.history)[["epoch", "fid", "mifid", "kid", "is"]]
    display(df)